# Countdown: дистилляция Qwen3-4B -> Gemma-3-1B

Teacher генерирует решения, student дообучается на них через SFT (4-bit + LoRA).
На инференсе ответ проверяется, если он неверный - решение ищется перебором.

In [ ]:
!pip install -q trl peft bitsandbytes accelerate datasets transformers huggingface_hub

In [ ]:
import multiprocessing
multiprocessing.set_start_method("spawn", force=True)

import os
import gc
import re
import ast
import math
from collections import Counter
from itertools import permutations, product

import torch
import pandas as pd

from datasets import load_dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login
from tqdm.auto import tqdm

In [ ]:
# токен из секретов Colab или из переменной окружения
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
login(HF_TOKEN)

TEACHER_NAME = "Qwen/Qwen3-4B-Instruct-2507"
STUDENT_NAME = "google/gemma-3-1b-it"

ADAPTER_PATH = "competition_gemma/final_adapter"
PREPARED_DATASET_PATH = "content/countdown_prepared_80k"

SKIP_TRAIN_IF_ADAPTER_EXISTS = True
TRAIN_FROM_SCRATCH = not (SKIP_TRAIN_IF_ADAPTER_EXISTS and os.path.exists(ADAPTER_PATH))

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
dataset = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "all", split="train")
dataset = dataset.shuffle(seed=42).select(range(80000))

print(dataset)

In [ ]:
torch.cuda.empty_cache()
gc.collect()

teacher_tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_NAME,
    padding_side="left",
    use_fast=True,
)

if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

teacher_model.eval()

In [ ]:
student_tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_NAME,
    padding_side="left",
    use_fast=True,
)

if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

student_model.config.use_cache = False  # иначе конфликтует с обучением

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    bias="none",
)

student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()

In [ ]:
def format_prompt(nums, target):
    return (
        "Ты помощник, который умеет подробно рассуждать в уме, "
        "а затем аккуратно формулировать окончательный ответ.\n"
        f"Используя числа {nums}, составь уравнение, которое даёт в результате {target}. "
        "Можно использовать только базовые операции (+, -, *, /), "
        "и каждое число можно использовать не более одного раза.\n"
        "Промежуточные рассуждения оформи внутри тегов <think></think>. "
        "Финальное уравнение и ответ верни в тегах <answer></answer>, "
        "например: <answer>1 + 2 + 3 = 6</answer>."
    )

def strip_parentheses(expr: str) -> str:
    return expr.replace("(", "").replace(")", "").strip()

def extract_equation_from_teacher(text: str) -> str:
    m = re.search(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        candidate = m.group(1).strip()
        if "=" in candidate:
            return strip_parentheses(candidate)

    # тега нет - берём первую строку с "="
    for line in text.splitlines():
        line = line.strip()
        if "=" in line:
            return strip_parentheses(line)

    return strip_parentheses(text.splitlines()[0]) if text.strip() else "0 = 0"

def batch_generate_teacher(batch):
    prompts = [format_prompt(nums, target) for nums, target in zip(batch["nums"], batch["target"])]

    inputs = teacher_tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=160,
    ).to(teacher_model.device)

    with torch.no_grad():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=48,
            do_sample=False,
            pad_token_id=teacher_tokenizer.pad_token_id,
            eos_token_id=teacher_tokenizer.eos_token_id,
            use_cache=True,
        )

    decoded = teacher_tokenizer.batch_decode(outputs, skip_special_tokens=True)

    texts = []
    for prompt, answer in zip(prompts, decoded):
        completion = answer[len(prompt):].strip() if answer.startswith(prompt) else answer.strip()
        texts.append(prompt + completion)

    return {"text": texts}

In [ ]:
SAVE_PREPARED_DATASET = True

if os.path.exists(PREPARED_DATASET_PATH):
    prepared_dataset = load_from_disk(PREPARED_DATASET_PATH)
    print(f"Загружаю готовый датасет из {PREPARED_DATASET_PATH}")
else:
    prepared_dataset = dataset.map(
        batch_generate_teacher,
        batched=True,
        batch_size=16,
        remove_columns=dataset.column_names,
        desc="Генерирую ответы",
    )

    if SAVE_PREPARED_DATASET:
        prepared_dataset.save_to_disk(PREPARED_DATASET_PATH)
        print(f"Сохранил {PREPARED_DATASET_PATH}")

print(prepared_dataset)
print(prepared_dataset[0]["text"][:500])

In [ ]:
if TRAIN_FROM_SCRATCH:
    sft_config = SFTConfig(
        output_dir="./competition_gemma",

        per_device_train_batch_size=16,
        gradient_accumulation_steps=4,  # batch 64

        learning_rate=1e-4,
        num_train_epochs=1,

        fp16=False,
        bf16=True,

        gradient_checkpointing=False,
        logging_steps=100,
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        dataloader_num_workers=2,
        remove_unused_columns=True,
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        warmup_steps=500,
        lr_scheduler_type="cosine",
        seed=42,
        max_length=128,
        packing=True,
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=student_model,
        args=sft_config,
        train_dataset=prepared_dataset,
        processing_class=student_tokenizer,
    )

    torch.cuda.empty_cache()
    gc.collect()

    trainer.train()

    trainer.model.save_pretrained(ADAPTER_PATH)
    student_tokenizer.save_pretrained(ADAPTER_PATH)

    print(f"Адаптер и токенизатор сохранены в {ADAPTER_PATH}")
else:
    print("Адаптер есть")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH,
    padding_side="left",
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False,
)

model.eval()

device = list(model.hf_device_map.keys())[0] if hasattr(model, "hf_device_map") else "cuda"
print("Устройство:", device)

student_model = model
student_tokenizer = tokenizer

In [ ]:
test_df = pd.read_csv("test_public.csv")

# nums в csv лежит строкой вида "[1, 2, 3]"
if "nums_parsed" not in test_df.columns:
    test_df["nums_parsed"] = test_df["nums"].apply(
        lambda s: [int(x) for x in ast.literal_eval(s)] if isinstance(s, str) else list(s)
    )
test_df.head()

In [ ]:
SYSTEM_PROMPT = (
    "Ты помощник, который сначала продумывает решение в уме, "
    "а потом формулирует аккуратный окончательный ответ."
)

USER_TEMPLATE = (
    "Используя числа {nums}, составь уравнение, которое даёт в результате {target}. "
    "Можно использовать только базовые арифметические операции (+, -, *, /), "
    "и каждое число можно использовать не более одного раза. "
    "Промежуточные рассуждения оформи в тегах <think></think>. "
    "Финальное уравнение и ответ верни в тегах <answer></answer>."
)

ANSWER_TAG_RE = re.compile(r"<answer>(.*?)</answer>", flags=re.IGNORECASE | re.DOTALL)

MATH_CHARS = set("0123456789+-*/= ")

def format_prompt_for_student(nums, target):
    user_part = USER_TEMPLATE.format(nums=nums, target=target)
    return f"{SYSTEM_PROMPT}\n{user_part}"

def strip_outer_prompt_text(text: str) -> str:
    t = text
    if SYSTEM_PROMPT in t:
        t = t.split(SYSTEM_PROMPT, 1)[-1]
    return t.strip()

def clean_equation_text(text: str) -> str:
    candidates = []
    for part in re.split(r"[\n\r]", text):
        part = part.strip()
        if any(op in part for op in ["+", "-", "*", "/", "="]) and any(ch.isdigit() for ch in part):
            candidates.append(part)

    if not candidates:
        return ""

    eq = candidates[-1]

    if "=" in eq:
        eq = eq.split("=")[0].strip()

    eq = "".join(ch for ch in eq if ch in MATH_CHARS)
    eq = re.sub(r"\s+", " ", eq).strip()

    return eq

def extract_equation_from_student(text: str) -> str:
    text = strip_outer_prompt_text(text)

    m = ANSWER_TAG_RE.search(text)
    if m:
        candidate = m.group(1).strip()
        candidate = re.sub(r"^\s*answer\s*", "", candidate, flags=re.IGNORECASE)
        cleaned = clean_equation_text(candidate)
        if cleaned:
            return cleaned

    for line in text.splitlines():
        line = line.strip()
        cleaned = clean_equation_text(line)
        if cleaned:
            return cleaned

    return ""

In [ ]:
# считаем слева направо, без приоритета операций
def eval_left_to_right_expr(expr: str):
    tokens = expr.split()
    if not tokens:
        return None

    try:
        value = float(tokens[0])
        i = 1
        while i < len(tokens) - 1:
            op = tokens[i]
            nxt = float(tokens[i + 1])

            if op == "+":
                value = value + nxt
            elif op == "-":
                value = value - nxt
            elif op == "*":
                value = value * nxt
            elif op == "/":
                if nxt == 0:
                    return None
                value = value / nxt
            else:
                return None

            i += 2

        return value
    except Exception:
        return None

def is_valid_equation(expr: str, nums, target, tol=1e-3):
    if not expr:
        return False

    tokens = expr.split()
    used = []

    for t in tokens:
        if t.isdigit():
            used.append(int(t))

    c_used = Counter(used)
    c_nums = Counter(nums)

    for k, v in c_used.items():
        if c_nums[k] < v:
            return False

    val = eval_left_to_right_expr(expr)
    if val is None:
        return False

    return abs(val - target) < tol

In [ ]:
def batch_generate_student(prompts, max_new_tokens=64):
    inputs = student_tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
    ).to(student_model.device)

    with torch.no_grad():
        outputs = student_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=student_tokenizer.pad_token_id,
            eos_token_id=student_tokenizer.eos_token_id,
            use_cache=True,
        )

    decoded = student_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return decoded

def generate_equations_for_test_df(df, batch_size=16):
    equations = []

    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]

        prompts = [
            format_prompt_for_student(nums, target)
            for nums, target in zip(batch["nums"], batch["target"])
        ]

        generations = batch_generate_student(prompts)

        for gen in generations:
            eq = extract_equation_from_student(gen)
            equations.append(eq)

    return equations

In [ ]:
OPS = ["+", "-", "*", "/"]

def find_equation_bruteforce(nums, target, max_len=None):
    n = len(nums)
    if max_len is None or max_len > n:
        max_len = n

    for length in range(max_len, 1, -1):
        for perm in permutations(nums, length):
            for ops in product(OPS, repeat=length - 1):
                parts = []
                for i in range(length):
                    parts.append(str(perm[i]))
                    if i < length - 1:
                        parts.append(ops[i])

                expr = " ".join(parts)
                val = eval_left_to_right_expr(expr)

                if val is None:
                    continue

                if abs(val - target) < 1e-3:
                    return expr

    return ""

In [ ]:
model_equations = generate_equations_for_test_df(test_df, batch_size=16)

final_equations = []
valid_flags = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    nums = row["nums_parsed"]
    target = row["target"]

    eq = model_equations[i]

    # модель ошиблась - ищем перебором
    if not is_valid_equation(eq, nums, target):
        eq = find_equation_bruteforce(nums, target, max_len=len(nums))

    if not eq:
        eq = "0"

    ok = is_valid_equation(eq, nums, target)
    final_equations.append(eq)
    valid_flags.append(ok)

print("Валидных решений:", sum(valid_flags), "/", len(valid_flags))

In [ ]:
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "equation": final_equations,
})

submission_df.to_csv("submission.csv", index=False)

submission_df.head()

In [ ]:
sub_check = pd.read_csv("submission.csv")

print("Размер:", sub_check.shape)
print("NaN в equation:", sub_check["equation"].isna().sum())
print(
    "Пустые строки в equation:",
    sub_check["equation"].fillna("").astype(str).str.strip().eq("").sum(),
)

print("\nПервые строки файла")
display(sub_check.head())

print("\nСлучайные 5 строк")
display(sub_check.sample(5, random_state=42))